In [1]:
import pandas as pd
import os

# UK Housing Prices vs. Mortgage Affordability (2005 – 2025)

## 1. Problem Formulation
This project explores the relationship between UK property values and buyer purchasing power. While the **UK House Price Index (HPI)** tracks market prices, the **Bank of England’s Mortgage Lenders and Administrators Statistics (MLAR)** reveals the underlying financial risk and affordability. The goal is to translate this economic problem into a mathematical counterpart to determine if mortgage metrics serve as indicators for house price shifts.

## 2. Project Objectives
* **Data Integration:** Consolidate monthly price data and quarterly lending statistics to demonstrate skills in merging and validating various data types.
* **Exploratory Data Analysis (EDA):** Use analytical graphs and plots to visualize 20-year trends and identify market cycles.
* **Mathematical Analysis:** Apply statistical methods and hypothesis testing to validate the link between lending constraints and price volatility.

## 3. Data Sources
* **HM Land Registry (UKHPI):** The primary source for housing price trends, providing average prices and sales volumes based on completed sales.
* **Bank of England (MLAR):** An independent data source used for mortgage affordability, covering Loan-to-Value (LTV) and Loan-to-Income (LTI) ratios.

In [6]:
# 1. Setup paths
data_folder = 'data'

# 2. Load the files
house_price_data = pd.read_csv(os.path.join(data_folder, 'ukhpi-united-kingdom-from-2005-01-01-to-2025-12-01.csv'))
mortgage_rate_data = pd.read_csv(os.path.join(data_folder, 'Bank of England  Database.csv'))
employment_data = pd.read_csv(os.path.join(data_folder, 'emp.csv'), skiprows=6)

# 3. Vetting
print("--- DATASET INTEGRITY CHECK ---")
print(f"HPI Rows: {house_price_data.shape[0]} | Rates Rows: {mortgage_rate_data.shape[0]}")
print(f"Date Alignment: {house_price_data['Period'].iloc[-1]} matches BoE Latest Date: {mortgage_rate_data['Date'].iloc[0]}")

--- DATASET INTEGRITY CHECK ---
HPI Rows: 252 | Rates Rows: 252
Date Alignment: 2025-12 matches BoE Latest Date: 31 Dec 25


In [4]:
# 1. Cleaning the Mortgage Rate Data
if 'rate_2yr' not in mortgage_rate_data.columns:
    mortgage_rate_data.rename(columns={
        mortgage_rate_data.columns[0]: 'date',
        mortgage_rate_data.columns[1]: 'rate_2yr',
        mortgage_rate_data.columns[3]: 'rate_5yr'
    }, inplace=True)
    mortgage_rate_data = mortgage_rate_data[['date', 'rate_2yr', 'rate_5yr']]

# 2. Cleaning the House Price Data
if 'avg_house_price' not in house_price_data.columns:
    house_price_data.rename(columns={
        'Period': 'date',
        'Average price All property types': 'avg_house_price'
    }, inplace=True)
    house_price_data = house_price_data[['date', 'avg_house_price']]

# 3. Cleaning the Employment Data
if 'real_earnings' not in employment_data.columns:
    employment_data.rename(columns={
        employment_data.columns[0]: 'date',
        employment_data.columns[4]: 'real_earnings'
    }, inplace=True)
    employment_data = employment_data[['date', 'real_earnings']]

# 4. Standardize all dates to the 1st of the month
for data in [mortgage_rate_data, house_price_data, employment_data]:
    data['date'] = pd.to_datetime(data['date'], errors='coerce', dayfirst=True)
    data.dropna(subset=['date'], inplace=True)
    data['date'] = data['date'].dt.to_period('M').dt.to_timestamp()

house_price_data = house_price_data[['date', 'avg_house_price']]

print("--- FINAL VERIFICATION ---")
print(f"Mortgage: {mortgage_rate_data.columns.tolist()}")
print(f"HPI: {house_price_data.columns.tolist()}")
print(f"Earnings: {employment_data.columns.tolist()}")

--- FINAL VERIFICATION ---
Mortgage: ['date', 'rate_2yr', 'rate_5yr']
HPI: ['date', 'avg_house_price']
Earnings: ['date', 'real_earnings']
